# StreamPulse: прогнозування відтоку (Churn)
**Мета:** заздалегідь виявити користувачів, які ймовірно скасують підписку, щоб запропонувати їм утримуючу пропозицію.

Пайплайн (Scikit-Learn):
1. Дані та EDA
2. Train/test split зі стратифікацією (до будь-якого препроцесингу, щоб уникнути data leakage)
3. `ColumnTransformer` + `Pipeline` (імп'ютація, масштабування, one-hot)
4. Порівняння моделей крос-валідацією
5. Підбір гіперпараметрів
6. Оцінка на тесті: ROC-AUC, PR-AUC, recall/precision
7. Вибір порогу за бізнес-вартістю
8. Важливість ознак і збереження моделі

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Дані
Реальної вибірки немає, тому генеруємо синтетичну (5000 користувачів, ~20% відтоку, з пропусками).
**Свої дані:** замініть клітинку на `df = pd.read_csv("your_file.csv")`, вкажіть назву цільової колонки в `TARGET`.

In [ ]:
n = 5000
df = pd.DataFrame({
    "customer_id": np.arange(n),
    "tenure_months": rng.integers(1, 61, n),
    "plan": rng.choice(["Basic", "Standard", "Premium"], n, p=[0.4, 0.4, 0.2]),
    "contract": rng.choice(["monthly", "annual"], n, p=[0.75, 0.25]),
    "device": rng.choice(["Mobile", "TV", "Web"], n, p=[0.45, 0.35, 0.2]),
    "payment_method": rng.choice(["card", "paypal", "carrier_billing"], n, p=[0.6, 0.25, 0.15]),
    "watch_hours_week": rng.gamma(2.0, 3.0, n),
    "sessions_30d": rng.poisson(14, n),
    "days_since_last_login": rng.exponential(6, n).round(),
    "support_tickets_90d": rng.poisson(0.7, n),
    "payment_failures_6m": rng.poisson(0.3, n),
    "has_discount": rng.integers(0, 2, n),
})
df["monthly_fee"] = df["plan"].map({"Basic": 6.99, "Standard": 11.99, "Premium": 17.99})

# Прихований механізм відтоку
logit = (-1.2
         - 0.04 * df["tenure_months"]
         + 1.0 * (df["contract"] == "monthly")
         - 0.12 * df["watch_hours_week"]
         - 0.03 * df["sessions_30d"]
         + 0.06 * df["days_since_last_login"]
         + 0.35 * df["support_tickets_90d"]
         + 0.6 * df["payment_failures_6m"]
         + 0.04 * df["monthly_fee"]
         - 0.4 * df["has_discount"]
         + rng.normal(0, 0.8, n))
df["churn"] = (rng.random(n) < 1 / (1 + np.exp(-logit - 0.6))).astype(int)

# Імітуємо реальні пропуски
for col in ["watch_hours_week", "sessions_30d", "payment_method"]:
    df.loc[rng.random(n) < 0.04, col] = np.nan

TARGET = "churn"
df.head()

In [ ]:
print(df.shape)
print("Частка відтоку:", df[TARGET].mean().round(3))
print("\nПропуски:\n", df.isna().sum()[lambda s: s > 0])
df.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df.groupby("contract")[TARGET].mean().plot.bar(ax=axes[0], title="Churn rate за типом контракту")
df.groupby("plan")[TARGET].mean().plot.bar(ax=axes[1], title="Churn rate за тарифом")
df.boxplot(column="days_since_last_login", by=TARGET, ax=axes[2])
axes[2].set_title("Днів від останнього входу"); plt.suptitle("")
plt.tight_layout(); plt.show()

## 2. Split і препроцесинг
Розбиваємо **до** будь-якого fit (стратифікація зберігає частку відтоку). Весь препроцесинг живе всередині `Pipeline`, тому під час крос-валідації нічого не «підглядає» у валідаційний фолд.
`customer_id` — ідентифікатор, у ознаки не потрапляє.

In [ ]:
X = df.drop(columns=[TARGET, "customer_id"])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Числові:", num_cols)
print("Категоріальні:", cat_cols)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

## 3. Порівняння моделей (стратифікована 5-fold CV)
Відтік — незбалансований клас, тому `accuracy` оманлива. Основні метрики: **ROC-AUC** та **PR-AUC (average precision)**. Для боротьби з дисбалансом використовуємо `class_weight="balanced"`.

In [ ]:
models = {
    "Baseline (dummy)": DummyClassifier(strategy="prior"),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                           min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision",
                                  "recall": "recall", "precision": "precision"},
                         n_jobs=-1)
    rows.append({"model": name, **{m: res[f"test_{m}"].mean() for m in ["roc_auc", "pr_auc", "recall", "precision"]},
                 "roc_auc_std": res["test_roc_auc"].std()})

cv_results = pd.DataFrame(rows).set_index("model").round(3).sort_values("pr_auc", ascending=False)
cv_results

## 4. Підбір гіперпараметрів (RandomizedSearchCV)

In [ ]:
best_name = cv_results.drop("Baseline (dummy)").index[0]
print("Найкраща модель за PR-AUC:", best_name)

param_spaces = {
    "LogisticRegression": {"clf__C": np.logspace(-3, 2, 30)},
    "RandomForest": {"clf__n_estimators": [200, 400, 600],
                     "clf__max_depth": [None, 6, 10, 16],
                     "clf__min_samples_leaf": [1, 3, 5, 10],
                     "clf__max_features": ["sqrt", 0.5]},
    "HistGradientBoosting": {"clf__learning_rate": [0.02, 0.05, 0.1],
                             "clf__max_depth": [None, 3, 5],
                             "clf__max_leaf_nodes": [15, 31, 63],
                             "clf__l2_regularization": [0.0, 0.1, 1.0],
                             "clf__min_samples_leaf": [20, 50, 100]},
}

base_pipe = Pipeline([("prep", preprocessor), ("clf", models[best_name])])
search = RandomizedSearchCV(base_pipe, param_spaces[best_name], n_iter=15,
                            scoring="average_precision", cv=cv,
                            n_jobs=-1, random_state=RANDOM_STATE, refit=True)
search.fit(X_train, y_train)

print("Найкращі параметри:", search.best_params_)
print("CV PR-AUC:", round(search.best_score_, 3))
best_model = search.best_estimator_

## 5. Фінальна оцінка на відкладеній тестовій вибірці
Тест використовується **один раз**, після завершення підбору.

In [ ]:
proba = best_model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
print(f"PR-AUC : {average_precision_score(y_test, proba):.3f}  (baseline = {y_test.mean():.3f})")

pred_default = (proba >= 0.5).astype(int)
print("\nЗвіт при порозі 0.5:\n", classification_report(y_test, pred_default, target_names=["stay", "churn"]))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0]); axes[0].set_title("ROC")
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1]); axes[1].set_title("Precision-Recall")
ConfusionMatrixDisplay.from_predictions(y_test, pred_default, ax=axes[2], colorbar=False); axes[2].set_title("Confusion matrix (поріг 0.5)")
plt.tight_layout(); plt.show()

## 6. Поріг рішення за бізнес-вартістю
Поріг 0.5 рідко оптимальний. Задаємо економіку кампанії (**змініть під реальні цифри**):
- `OFFER_COST` — вартість пропозиції одному користувачу;
- `SAVED_VALUE` — вигода від утриманого клієнта (LTV);
- `ACCEPT_RATE` — частка тих, хто, отримавши пропозицію, залишається.

Очікуваний прибуток = TP·(ACCEPT_RATE·SAVED_VALUE − OFFER_COST) − FP·OFFER_COST.

In [ ]:
OFFER_COST = 5.0
SAVED_VALUE = 120.0
ACCEPT_RATE = 0.30

thresholds = np.linspace(0.05, 0.95, 91)
profits = []
for t in thresholds:
    pred = proba >= t
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    profits.append(tp * (ACCEPT_RATE * SAVED_VALUE - OFFER_COST) - fp * OFFER_COST)

best_t = thresholds[int(np.argmax(profits))]
print(f"Оптимальний поріг: {best_t:.2f}, очікуваний прибуток на тесті: {max(profits):.0f}")

plt.plot(thresholds, profits)
plt.axvline(best_t, color="red", ls="--", label=f"поріг {best_t:.2f}")
plt.xlabel("Поріг"); plt.ylabel("Очікуваний прибуток"); plt.title("Вибір порогу за прибутком")
plt.legend(); plt.show()

pred_best = (proba >= best_t).astype(int)
print(classification_report(y_test, pred_best, target_names=["stay", "churn"]))

## 7. Важливість ознак (permutation importance на сирих ознаках)

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring="average_precision",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

imp.plot.barh(xerr=pd.Series(perm.importances_std, index=X_test.columns)[imp.index])
plt.title("Permutation importance (падіння PR-AUC)"); plt.tight_layout(); plt.show()

## 8. Збереження пайплайну та приклад застосування

In [ ]:
joblib.dump({"model": best_model, "threshold": float(best_t)}, "churn_model.joblib")

artifact = joblib.load("churn_model.joblib")
risk = artifact["model"].predict_proba(X_test)[:, 1]
top = X_test.assign(churn_risk=risk, will_receive_offer=risk >= artifact["threshold"]) \
            .sort_values("churn_risk", ascending=False)
top.head(10)

## Висновки
- Весь препроцесинг та модель зібрані в єдиний `Pipeline`, тому CV та продакшн використовують ту саму логіку без витоку даних.
- Через дисбаланс класів оцінюємо моделі за **PR-AUC і ROC-AUC**, а не accuracy; baseline показує, наскільки модель кращa за випадкове вгадування.
- Поріг класифікації обрано за **очікуваним прибутком кампанії**, а не за замовчуванням 0.5; його варто перераховувати при зміні вартості пропозиції чи LTV.
- Найважливіші ознаки (див. permutation importance) підказують, на які сигнали (давність останнього входу, тип контракту, збої оплати) звертати увагу команди утримання.
- **Наступні кроки:** валідація за часом (train на минулому, test на майбутньому), калібрування ймовірностей (`CalibratedClassifierCV`), моніторинг дрейфу даних, A/B-тест пропозицій.